# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.3 of 7 — Extract Classification Results

**Role in the pipeline:**
Reads the Batch API output JSONL files downloaded by notebook 3.2, parses the function-call arguments to extract the LLM-assigned ESCO occupation code and title, and merges them into the Stage 2 output DataFrames. The resulting pickles (with `esco_code` and `esco_title` columns added) are saved to `STAGE3_RESULT_PATH`.

Also computes per-file statistics on how many vacancies received a valid classification and logs them to the tracker. This information is used by notebook 3.4 to separate complete from missing records.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. 3.2 — Submit batch jobs and monitor status
3. ▶ **3.3 — Extract classification results** ← *you are here*
4. 3.4 — Split missing and complete classifications
5. 3.5 — Create batch input files for missing records
6. 3.6 — Submit batch jobs for missing records *(run after Batch API completes)*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Notebook 3.2 completed — `output_batch_status == 'complete'` for all rows

## 3.3.1. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules and pandas.

In [2]:
import stage2 as st2
import stage3 as st3
import pandas as pd

## 3.3.2. Load stage 3 logs

Reads the Stage 3 tracker. The commented-out lines show how to reset `result_status` and `result_path` columns for a full re-extraction from scratch (use only if the result pickles need to be regenerated).

Loads the tracker, with optional commented-out reset lines for `result_status` and `result_path`.

In [3]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df

,input_file,extract_path,input_batch_path,input_batch_status,job_id,job_status,output_batch_path,output_batch_status,result_path,result_status
0,ua-2024-01-01,../data/stage_02/processed/output\ua-2024-01-0...,../data/stage_03/raw/input\ua-2024-01-01.jsonl,created,batch_69c055917b0481909b82ddb32c228250,completed,../data/stage_03/processed/output\ua-2024-01-0...,complete,NaN,NaN


## 3.3.3. Read output batches and add data to results

Extraction loop: for each row where `output_batch_status == 'complete'` but `result_status != 'complete'`:

1. Loads the Stage 2 output pickle (all vacancy columns).
2. Calls `extract_esco_codes()` to parse the Batch API output JSONL and return a DataFrame with `id`, `esco_code`, `esco_title`.
3. Merges the ESCO columns onto the Stage 2 data via a left-join on `id`. Vacancies whose request failed will have `NaN` in the ESCO columns (these become the "missing" records handled by notebooks 3.4–3.7).
4. Saves the merged pickle to `STAGE3_RESULT_PATH` and marks `result_status = 'complete'`.

In [4]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)

for _,row in process_df.iterrows():
    if row["output_batch_status"] == "complete" and row["result_status"] != "complete":
        result_df = pd.read_pickle(row["extract_path"])
        esco_df = st3.extract_esco_codes(row["output_batch_path"])
        result_df['id'] = result_df['id'].astype(str)

        result_df = pd.merge(result_df, esco_df, on='id', how='left')

        result_path = os.path.join(g.Config.STAGE3_RESULT_PATH, row["input_file"]  + ".pkl")
        result_df.to_pickle(result_path)
        process_df.loc[_, "result_path"] = result_path
        process_df.loc[_, "result_status"] = "complete"

        process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)
        print("Added result file for: ", row["input_file"])

../data/stage_03/processed/output\ua-2024-01-01.json
Added result file for:  ua-2024-01-01


Displays the last result DataFrame extracted by the loop above — useful for a quick sanity check on the merged columns.

In [5]:
result_df

,id,title,description,min_salary,max_salary,currency,salary_rate,date_created,date_expired,clean_title,clean_desc,title_lang,desc_lang,skill_ids,skill_labels,job_type,job_type_score,esco_code,esco_title
0,2673941879650338041,Accountant,Our company is seeking a motivated individual ...,50000.0,57000.0,EUR,hourly,2024-01-01,2024-02-26,accountant,our company is seeking a motivated individual ...,en,en,http://data.europa.eu/esco/skill/4ddf5204-95f5...,"apply business acumen,motivate employees,asses...",combined,0.4700,2411,Accountants
1,2829269335131763744,Accountant,We are expanding our team and looking for a de...,NaN,NaN,None,None,2024-01-01,2024-01-30,accountant,we are expanding our team and looking for a de...,en,en,http://data.europa.eu/esco/skill/652da95d-b336...,"communicate technicalities with clients,liaise...",in_office,0.4894,2411,Accountants
2,2292953494970979965,Accountant,We are looking for an experienced professional...,21000.0,40000.0,EUR,monthly,2024-01-01,2024-02-19,accountant,we are looking for an experienced professional...,en,en,http://data.europa.eu/esco/skill/1212ac42-2564...,"advise on personnel management,cooperate with ...",in_office,0.4967,2411,Accountants
3,8991901255055716088,Backend Developer,Looking for a proactive team player to support...,NaN,NaN,None,None,2024-01-01,2024-02-07,backend developer,looking for a proactive team player to support...,en,en,http://data.europa.eu/esco/skill/75e48971-3242...,"manage teamwork,work in teams,manage a team,co...",combined,0.4720,2132,Software developers
4,9937412981206494120,Business Analyst,Opportunity to work with cutting-edge technolo...,NaN,NaN,None,None,2024-01-01,2024-02-14,business analyst,opportunity to work with cutting-edge technolo...,en,en,http://data.europa.eu/esco/skill/8327754d-3990...,"innovate in ICT,seek innovation in current pra...",in_office,0.4277,2421,Business analysts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5682536959350677812,Warehouse Manager,Exciting opportunity for a skilled professiona...,NaN,NaN,None,None,2024-01-01,2024-02-16,warehouse manager,exciting opportunity for a skilled professiona...,en,en,http://data.europa.eu/esco/skill/885564e4-9818...,"managing a business with great care,communicat...",combined,0.4900,1324,Warehouse manager
96,4745382362973408538,Warehouse Manager,Looking for a proactive team player to support...,NaN,NaN,None,None,2024-01-01,2024-02-21,warehouse manager,looking for a proactive team player to support...,en,en,http://data.europa.eu/esco/skill/75e48971-3242...,"manage teamwork,work in teams,manage a team,co...",combined,0.4246,1330,Warehouse manager
97,1748822233724604818,Warehouse Manager,Seeking an experienced professional with stron...,NaN,NaN,UAH,monthly,2024-01-01,2024-02-28,warehouse manager,seeking an experienced professional with stron...,en,en,http://data.europa.eu/esco/skill/677d5144-4c77...,"own management skills,leadership principles,ad...",in_office,0.4480,1324,Warehouse manager
98,5942472651725435914,Warehouse Manager,We are expanding our team and looking for a de...,NaN,NaN,UAH,hourly,2024-01-01,2024-02-17,warehouse manager,we are expanding our team and looking for a de...,en,en,http://data.europa.eu/esco/skill/652da95d-b336...,"communicate technicalities with clients,liaise...",combined,0.4383,1324,Warehouse manager


## 3.3.4. Check completion statistics

Prints a summary of the processing pipeline progress across all tracked files:
- Batch input files created
- API jobs completed
- Output batch files downloaded
- Result pickles extracted

All four counts should reach 100% before proceeding to notebook 3.4.

In [6]:
input_created= (process_df["input_batch_status"] == "created").sum()
t = process_df.shape[0]
jobs_completed = (process_df["job_status"] == "completed").sum()
output_complete = (process_df["output_batch_status"] == "complete").sum()
result_complete = (process_df["result_status"] == "complete").sum()

print(f"Total created input batches {input_created} of {t}:\t\t{round(100*input_created/t, 2)} %")
print(f"Total completed jobs {jobs_completed} of {t}:\t\t\t{round(100*jobs_completed/t, 2)} %")
print(f"Total created output batches {output_complete} of {t}:\t{round(100*output_complete/t, 2)} %")
print(f"Total complete results {result_complete} of {t}:\t\t\t{round(100*result_complete/t, 2)} %")

Total created input batches 1 of 1:		100.0 %
Total completed jobs 1 of 1:			100.0 %
Total created output batches 1 of 1:	100.0 %
Total complete results 1 of 1:			100.0 %


Per-file missing count check: iterates over all result pickles, counts how many vacancies have a missing or very short `esco_title` (length < 3), and stores the `missing` count, `total` count, and `missing_percent` in a summary DataFrame. These become the records targeted by notebooks 3.5–3.7.

In [7]:
stats_df = pd.DataFrame(
    {"input_file": process_df["input_file"], "result_path": process_df["result_path"], "missing": None, "total": None,
     "missing_percent": None})

for i, row in stats_df.iterrows():
    print("Working on: ", row["input_file"], "")
    try:
        df = pd.read_pickle(row["result_path"])
    except Exception:
        print(f"Error reading file {row['result_path']}")
        continue

    total = len(df)
    if total == 0:
        stats_df.loc[i, ["total", "missing", "missing_percent"]] = [0, 0, None]
        continue

    if "esco_title" in df.columns:
        ser = df["esco_title"]
        missing_mask = ser.isna() | (ser.astype(str).str.len() < 3)
        missing = int(missing_mask.sum())
    else:
        missing = total  # if column absent, consider all missing

    stats_df.loc[i, "total"] = total
    stats_df.loc[i, "missing"] = missing
    stats_df.loc[i, "missing_percent"] = (missing / total) if total else None

stats_df

Working on:  ua-2024-01-01 


,input_file,result_path,missing,total,missing_percent
0,ua-2024-01-01,../data/stage_03/processed/result\ua-2024-01-0...,0,100,0.0


Shows the column names present in the tracker at this point — useful for verifying which columns are available for the next stage.

In [8]:
process_df.columns

Index(['input_file', 'extract_path', 'input_batch_path', 'input_batch_status',
       'job_id', 'job_status', 'output_batch_path', 'output_batch_status',
       'result_path', 'result_status'],
      dtype='object')

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.